In [1]:
from sentence_transformers import SentenceTransformer
from pydantic import BaseModel
from typing import List
from scipy.spatial import distance
from openai import OpenAI
from dotenv import load_dotenv
import numpy as np
import os
import re

load_dotenv()

client_emb = OpenAI(
    api_key=os.getenv('LOCAL_TOKEN'),
    base_url=os.getenv('LOCAL_URL')
)

client_llm = OpenAI(
    api_key=os.getenv('GROQ_TOKEN'),
    base_url=os.getenv('BASE_URL_GROQ')
)

articles = [
    {"headline": "SpaceX Launches New Starship Rocket",
     "topic": "Science",
     "keywords": ["space", "rocket", "nasa", "exploration"]},
    {"headline": "Bitcoin Reaches All-Time High as Investors Rush In",
     "topic": "Business",
     "keywords": ["crypto", "bitcoin", "finance", "investment"]},
    {"headline": "Champions League Final: Real Madrid vs Manchester City",
     "topic": "Sport",
     "keywords": ["football", "soccer", "champions league", "madrid"]},
    {"headline": "New AI Model Outperforms Humans in Medical Diagnosis",
     "topic": "Tech",
     "keywords": ["ai", "healthcare", "machine learning", "diagnosis"]},
    {"headline": "Global Leaders Meet to Discuss Climate Change",
     "topic": "Science",
     "keywords": ["climate", "environment", "policy", "emissions"]},
    {"headline": "Apple Announces Revolutionary New iPhone",
     "topic": "Tech",
     "keywords": ["apple", "iphone", "smartphone", "technology"]},
    {"headline": "Stock Markets Plunge Amid Recession Fears",
     "topic": "Business",
     "keywords": ["stocks", "economy", "recession", "wall street"]},
    {"headline": "Olympics 2024: USA Wins Gold in Swimming",
     "topic": "Sport",
     "keywords": ["olympics", "swimming", "usa", "gold medal"]},
    {"headline": "Scientists Discover New Species in Amazon Rainforest",
     "topic": "Science",
     "keywords": ["biology", "amazon", "species", "nature"]},
    {"headline": "Tesla Unveils Autonomous Robot for Home Use",
     "topic": "Tech",
     "keywords": ["tesla", "robot", "automation", "ai"]},
    {"headline": "Federal Reserve Raises Interest Rates Again",
     "topic": "Business",
     "keywords": ["economy", "interest rates", "fed", "banking"]},
    {"headline": "NBA Finals: LeBron James Leads Team to Victory",
     "topic": "Sport",
     "keywords": ["basketball", "nba", "lebron", "finals"]},
]

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Часть 1
## Задача 1.1

def create_article_text(article):
    return f'''Headline: {article['headline']}
Topic: {article['topic']}
Keywords: {', '.join(article['keywords'])}'''

print(create_article_text(articles[0]))
print()
print(create_article_text(articles[-1]))

Headline: SpaceX Launches New Starship Rocket
Topic: Science
Keywords: space, rocket, nasa, exploration

Headline: NBA Finals: LeBron James Leads Team to Victory
Topic: Sport
Keywords: basketball, nba, lebron, finals


In [3]:
## Задача 1.2

def create_embeddings(texts):
    response = client_emb.embeddings.create(
        model=os.getenv('LOCAL_MODEL'),
        input=texts
    )

    response_dict = response.model_dump()

    return [data['embedding'] for data in response_dict['data']]

article_texts = [create_article_text(article) for article in articles]
article_embeddings = create_embeddings(article_texts)

print(f'''Создано эмбеддингов: {len(article_embeddings)}
Длина каждого вектора: {len(article_embeddings[0])}''')

Создано эмбеддингов: 12
Длина каждого вектора: 1024


In [5]:
# Часть 2
## Задача 2.1

def find_n_closest(query_vector:list, embeddings:list[list], n:int=3):
    distances = []
    for i, emb in enumerate(embeddings):
        dist = distance.cosine(query_vector, emb)
        distances.append({'distance':dist, 'index':i})
    
    sorted_distances = sorted(distances, key=lambda x: x['distance'], )
    filtered_distances = sorted_distances[:n]

    return filtered_distances

In [8]:
## Задача 2.2
queries = [
    "artificial intelligence and technology",
    "financial markets and economy",
    "athletes and sports competitions",
]

for query in queries:
    print(f'Запрос: "{query}"')
    query_embedding = create_embeddings(query)[0]

    hits = find_n_closest(query_embedding, article_embeddings)

    for i, hit in enumerate(hits):
        print(f'    {i+1}. {articles[hit['index']]['headline']} (dist={round(hit['distance'], 4)})')
    print()

Запрос: "artificial intelligence and technology"
    1. New AI Model Outperforms Humans in Medical Diagnosis (dist=0.4031)
    2. Tesla Unveils Autonomous Robot for Home Use (dist=0.4225)
    3. Apple Announces Revolutionary New iPhone (dist=0.526)

Запрос: "financial markets and economy"
    1. Stock Markets Plunge Amid Recession Fears (dist=0.4967)
    2. Federal Reserve Raises Interest Rates Again (dist=0.5049)
    3. Bitcoin Reaches All-Time High as Investors Rush In (dist=0.5529)

Запрос: "athletes and sports competitions"
    1. Olympics 2024: USA Wins Gold in Swimming (dist=0.4118)
    2. Champions League Final: Real Madrid vs Manchester City (dist=0.5223)
    3. NBA Finals: LeBron James Leads Team to Victory (dist=0.5316)



In [9]:
# Часть 3
## Задача 3.1

current_article = {
    "headline": "Ethereum Surges 30% Following Network Upgrade",
    "topic": "Business",
    "keywords": ["ethereum", "crypto", "blockchain", "investment"]
}

print(f'Статья: "{current_article['headline']}"')
print('\nРекомендации:')

current_article_text = create_article_text(current_article)
current_article_embeddings = create_embeddings(current_article_text)[0]

hits = find_n_closest(current_article_embeddings, article_embeddings)

for i, hit in enumerate(hits):
    print(f'    {i+1}. {articles[hit['index']]['headline']} (dist={round(hit['distance'], 4)})')

Статья: "Ethereum Surges 30% Following Network Upgrade"

Рекомендации:
    1. Bitcoin Reaches All-Time High as Investors Rush In (dist=0.2774)
    2. Federal Reserve Raises Interest Rates Again (dist=0.4392)
    3. Stock Markets Plunge Amid Recession Fears (dist=0.5394)


In [10]:
# Часть 4
## Задача 4.1

user_history = [
    {"headline": "New AI Model Outperforms Humans in Medical Diagnosis",
     "topic": "Tech",
     "keywords": ["ai", "healthcare", "machine learning", "diagnosis"]},
    {"headline": "Apple Announces Revolutionary New iPhone",
     "topic": "Tech",
     "keywords": ["apple", "iphone", "smartphone", "technology"]},
]

user_history_texts = [create_article_text(article) for article in user_history]
user_history_embeddings = create_embeddings(user_history_texts)
user_history_embeddings_mean = np.mean(user_history_embeddings, axis = 0)

print(f'''Эмбеддингов в истории: {len(user_history_embeddings)}
Размерность среднего вектора: {len(user_history_embeddings_mean)}''')

Эмбеддингов в истории: 2
Размерность среднего вектора: 1024


In [11]:
## Часть 4.2

filtered_articles = [article for article in articles if article not in user_history]

filtered_article_text = [create_article_text(article) for article in filtered_articles]
filtered_article_embeddings = create_embeddings(filtered_article_text)

hits = find_n_closest(user_history_embeddings_mean, filtered_article_embeddings)

print(f'''Статей в каталоге: {len(articles)}
Статей после фильтрации: {len(filtered_articles)}

Рекомендации (на основе истории):''')

for i, hit in enumerate(hits):
    print(f'    {i+1}. {filtered_articles[hit['index']]['headline']} (dist={round(hit['distance'], 4)})')

Статей в каталоге: 12
Статей после фильтрации: 10

Рекомендации (на основе истории):
    1. Tesla Unveils Autonomous Robot for Home Use (dist=0.3022)
    2. SpaceX Launches New Starship Rocket (dist=0.3836)
    3. Scientists Discover New Species in Amazon Rainforest (dist=0.4403)


In [14]:
# Часть 5
## Часть 5.1

def find_closest(query_vector, embeddings):
    distances = []
    for i, emb in enumerate(embeddings):
        dist = distance.cosine(query_vector, emb)
        distances.append({'distance':dist, 'index':i})

    return min(distances, key=lambda x: x['distance'])

topics = [
    {'label': 'Tech'},
    {'label': 'Science'},
    {'label': 'Sport'},
    {'label': 'Business'}
]

texts = [topic['label'] for topic in topics]
texts_embeddings = create_embeddings(texts)

def create_article_text_short(article):
    return f'''Headline: {article['headline']}
Keywords: {', '.join(article['keywords'])}'''

test_articles = [
    {"headline": "Quantum Computing Breakthrough Could Revolutionize Cryptography",
     "keywords": ["quantum", "computing", "encryption", "technology"]},
    {"headline": "Marathon World Record Broken at Berlin Race",
     "keywords": ["running", "marathon", "athletics", "world record"]},
    {"headline": "Startup Raises $500M to Build Nuclear Fusion Reactor",
     "keywords": ["energy", "nuclear", "startup", "investment"]},
]

test_article_text = [create_article_text_short(article) for article in test_articles]
test_article_embeddings = create_embeddings(test_article_text)

for i, article in enumerate(test_article_embeddings):
    closest = find_closest(article, texts_embeddings)
    print(f'{test_articles[i]['headline']} -> Topic: {topics[closest['index']]['label']}')


Quantum Computing Breakthrough Could Revolutionize Cryptography -> Topic: Tech
Marathon World Record Broken at Berlin Race -> Topic: Sport
Startup Raises $500M to Build Nuclear Fusion Reactor -> Topic: Business


In [18]:
## Задача 5.2

topics = [
    {'label': 'Tech',     'description': 'A news article about technology, gadgets, software or digital innovation'},
    {'label': 'Science',  'description': 'A news article about scientific research, discoveries or natural phenomena'},
    {'label': 'Sport',    'description': 'A news article about sports, athletes, competitions or championships'},
    {'label': 'Business', 'description': 'A news article about economy, finance, markets or investments'}
]

texts = [topic['description'] for topic in topics]
texts_embeddings = create_embeddings(texts)

print('=== Классификация с развёрнутыми описаниями ===')
for i, article in enumerate(test_article_embeddings):
    closest = find_closest(article, texts_embeddings)
    print(f'"{test_articles[i]['headline']}" -> {topics[closest['index']]['label']}')

print('''\n=== Сравнение ===
Метки совпали: 3 из 3''')

=== Классификация с развёрнутыми описаниями ===
"Quantum Computing Breakthrough Could Revolutionize Cryptography" -> Tech
"Marathon World Record Broken at Berlin Race" -> Sport
"Startup Raises $500M to Build Nuclear Fusion Reactor" -> Business

=== Сравнение ===
Метки совпали: 3 из 3


In [27]:
# Задача 6

def classify_article(article, topics, articles):
    classes = [topic['description'] for topic in topics]
    classes_embeddings = create_embeddings(classes)
    article_text = create_article_text(article)
    article_embedding = create_embeddings(article_text)[0]

    closest = find_closest(article_embedding, classes_embeddings)

    print(f'Статья: "{article['headline']}"')
    print(f'Определенная тема: {topics[closest['index']]['label']}')

    print(f'\nРекомендации в теме {topics[closest['index']]['label']}:')

    filtered_articles = [article for article in articles if article['topic'] == topics[closest['index']]['label']]
    article_texts = [create_article_text(article) for article in filtered_articles]
    article_embeddings = create_embeddings(article_texts)

    hits = find_n_closest(article_embedding, article_embeddings)

    for i, hit in enumerate(hits):
        print(f'    {i+1}. {filtered_articles[hit['index']]['headline']} (dist={round(hit['distance'],4)})')


article = {
    "headline": "Ethereum Surges 30% Following Network Upgrade",
    "topic": "Business",
    "keywords": ["ethereum", "crypto", "blockchain", "investment"]
}

classify_article(article, topics, articles)

Статья: "Ethereum Surges 30% Following Network Upgrade"
Определенная тема: Business

Рекомендации в теме Business:
    1. Bitcoin Reaches All-Time High as Investors Rush In (dist=0.2774)
    2. Federal Reserve Raises Interest Rates Again (dist=0.4392)
    3. Stock Markets Plunge Amid Recession Fears (dist=0.5394)
